## Finance Dataset Generator

Generates a synthetic **finance** dataset in Unity Catalog with **realistic statistical distributions** and Faker-generated PII.

### Parameters

| Parameter | Default | Description |
| --- | --- | --- |
| `catalog` | `industry_sample_data` | Target Unity Catalog |

### Tables

| Schema | Entity Table | Rows | Event Table | Rows | Key Features |
| --- | --- | --- | --- | --- | --- |
| `finance` | `accounts` | ~2K | `transactions` | 100K-500K | 7 account types, 7 currencies, 40+ merchants, pattern-based fraud flags |

### How to Run

1. Set the `catalog` widget parameter at the top of the notebook.
2. **Run All** — the notebook creates the `finance` schema and generates both tables.
3. Final cells apply column comments and a `RemoveAfter` tag for workspace retention compliance.


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);

DROP SCHEMA IF EXISTS IDENTIFIER(:catalog || '.finance') CASCADE;

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.finance');

In [0]:
%pip install faker --quiet

In [0]:
%restart_python

In [0]:
import random
import math
from datetime import datetime, timedelta, date
from faker import Faker
from pyspark.sql import Row

fake = Faker()
Faker.seed(42)
random.seed(42)
CATALOG = dbutils.widgets.get('catalog')
SCHEMA = "finance"
CATALOG_SCHEMA = f"{CATALOG}.{SCHEMA}"

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

# --- Accounts table (~2,000 rows) ---
account_types = ["Checking", "Savings", "Credit Card", "Investment", "Loan", "Money Market", "CD"]
statuses = ["Active", "Closed", "Frozen", "Dormant"]
currencies = ["USD", "EUR", "GBP", "CAD", "AUD", "JPY", "CHF"]
currency_weights = [55, 12, 8, 8, 5, 7, 5]
branches = [
    "New York", "Chicago", "San Francisco", "Miami", "Boston", "Dallas", "Seattle",
    "Los Angeles", "Denver", "Atlanta", "Philadelphia", "Houston", "Portland",
    "Minneapolis", "Charlotte", "Phoenix", "Detroit", "Tampa"]

balance_params = {
    "Checking":     (7.8, 1.2, 50, 50000),
    "Savings":      (8.8, 1.5, 100, 500000),
    "Credit Card":  (6.5, 1.3, 50, 25000),
    "Investment":   (10.2, 1.4, 500, 2000000),
    "Loan":         (9.2, 0.9, 1000, 500000),
    "Money Market": (9.5, 1.3, 1000, 1000000),
    "CD":           (9.0, 1.0, 500, 500000),
}

NUM_ACCOUNTS = 2000
accounts = []
for i in range(1, NUM_ACCOUNTS + 1):
    open_date = date(2018, 1, 1) + timedelta(days=random.randint(0, 2800))
    acct_type = random.choices(account_types, weights=[25, 20, 20, 15, 10, 5, 5])[0]
    mu, sigma, lo, hi = balance_params[acct_type]
    raw_balance = random.lognormvariate(mu, sigma)
    balance = float(round(clamp(raw_balance, lo, hi), 2))
    if acct_type in ("Credit Card", "Loan"):
        balance = -balance
    accounts.append(Row(
        account_id=i,
        customer_name=fake.name(),
        account_type=acct_type,
        balance=balance,
        currency=random.choices(currencies, weights=currency_weights)[0],
        status=random.choices(statuses, weights=[80, 8, 5, 7])[0],
        opened_date=open_date,
        branch=random.choice(branches)
    ))

account_lookup = {a.account_id: a for a in accounts}

accounts_df = spark.createDataFrame(accounts)
accounts_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.accounts")
print(f"✔ Created {CATALOG_SCHEMA}.accounts ({accounts_df.count()} rows)")

# --- Transactions table (randomized ~100K-500K rows) ---
categories = ["Payroll", "Transfer", "Purchase", "Refund", "Fee", "Interest", "Withdrawal", "Deposit", "Payment", "Dividend"]
account_txn_weights = {
    "Checking":     [20, 15, 25, 5, 5, 1, 10, 10, 8, 1],
    "Savings":      [2, 20, 1, 1, 3, 30, 1, 30, 2, 10],
    "Credit Card":  [0, 2, 45, 10, 8, 5, 0, 0, 25, 0],
    "Investment":   [0, 15, 2, 1, 5, 5, 10, 15, 2, 45],
    "Loan":         [0, 5, 0, 0, 5, 10, 0, 5, 70, 5],
    "Money Market": [0, 20, 1, 1, 3, 25, 5, 30, 5, 10],
    "CD":           [0, 10, 0, 0, 5, 50, 0, 25, 5, 5],
}

category_merchants = {
    "Purchase": [
        "Amazon", "Walmart", "Target", "Starbucks", "Shell", "Apple", "Netflix", "Uber",
        "Costco", "Home Depot", "Whole Foods", "Delta Airlines", "Hilton", "Best Buy",
        "Chevron", "McDonald's", "Spotify", "Airbnb", "Lyft", "Instacart",
        "DoorDash", "Nike", "Zara", "IKEA", "Walgreens", "CVS", "Trader Joe's",
        "Kroger", "Safeway", "Nordstrom", "Macy's", "Gap", "Sephora", "Lowe's",
        "BP", "ExxonMobil", "Chipotle", "Panera", "Grubhub", "Hulu"],
    "Refund":   ["Amazon", "Walmart", "Target", "Apple", "Costco", "Home Depot",
                 "Best Buy", "Nordstrom", "Macy's", "IKEA", "Nike", "Sephora"],
    "Fee":      ["Bank Fee", "ATM Fee", "Overdraft Fee", "Wire Fee", "Late Payment Fee",
                 "Foreign Transaction Fee", "Monthly Maintenance Fee", "Account Analysis Fee"],
    "Payment":  ["Utility Co", "AT&T", "Verizon", "Insurance Co", "Mortgage Svc",
                 "T-Mobile", "Comcast", "Duke Energy", "State Farm", "Progressive",
                 "Geico", "Water Dept", "Gas Company", "Student Loan Svc"],
}

channel_by_category = {
    "Payroll":     [5, 5, 0, 5, 85],
    "Transfer":    [30, 10, 0, 35, 25],
    "Purchase":    [35, 5, 0, 50, 10],
    "Refund":      [40, 5, 0, 45, 10],
    "Fee":         [70, 20, 5, 5, 0],
    "Interest":    [90, 5, 0, 5, 0],
    "Withdrawal":  [5, 15, 65, 10, 5],
    "Deposit":     [20, 25, 30, 15, 10],
    "Payment":     [35, 10, 0, 40, 15],
    "Dividend":    [80, 5, 0, 10, 5],
}
channels = ["Online", "Branch", "ATM", "Mobile App", "Wire"]

txn_amount_params = {
    "Payroll":     (7.8, 0.6, 500, 25000),
    "Transfer":    (5.8, 1.5, 10, 100000),
    "Purchase":    (3.5, 1.2, 1, 5000),
    "Refund":      (3.2, 1.0, 1, 3000),
    "Fee":         (2.5, 0.8, 1, 500),
    "Interest":    (1.5, 1.2, 0.01, 5000),
    "Withdrawal":  (4.6, 1.0, 20, 10000),
    "Deposit":     (6.4, 1.3, 50, 50000),
    "Payment":     (5.3, 1.1, 10, 25000),
    "Dividend":    (4.2, 1.4, 1, 50000),
}

transactions = []
NUM_EVENT_RECORDS = random.randint(100_000, 500_000)
for i in range(1, NUM_EVENT_RECORDS + 1):
    txn_ts = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 730), hours=random.randint(0, 23), minutes=random.randint(0, 59))
    acct_id = random.randint(1, NUM_ACCOUNTS)
    acct = account_lookup[acct_id]
    weights = account_txn_weights[acct.account_type]
    cat = random.choices(categories, weights=weights)[0]

    is_debit = cat in ["Purchase", "Fee", "Withdrawal", "Payment"]
    mu, sigma, lo, hi = txn_amount_params[cat]
    amount = float(round(clamp(random.lognormvariate(mu, sigma), lo, hi), 2))

    merchant = None
    if cat in category_merchants:
        merchant = random.choice(category_merchants[cat])

    ch_weights = channel_by_category.get(cat, [20, 20, 20, 20, 20])
    channel = random.choices(channels, weights=ch_weights)[0]

    # Fraud: pattern-based (large amounts, unusual channels, velocity)
    fraud_prob = 0.005
    if amount > 2000: fraud_prob += 0.02
    if amount > 5000: fraud_prob += 0.03
    if cat == "Purchase" and channel == "Wire": fraud_prob += 0.04
    if cat == "Withdrawal" and amount > 3000: fraud_prob += 0.02
    fraud_suspected = random.random() < fraud_prob

    transactions.append(Row(
        transaction_id=1000 + i,
        account_id=acct_id,
        transaction_date=txn_ts,
        amount=amount,
        transaction_type="Debit" if is_debit else "Credit",
        category=cat,
        merchant=merchant,
        channel=channel,
        currency=acct.currency,
        description=f"{cat} - ref {random.randint(100000, 999999)}",
        fraud_suspected=fraud_suspected
    ))

transactions_df = spark.createDataFrame(transactions)
transactions_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.transactions")
print(f"✔ Created {CATALOG_SCHEMA}.transactions ({transactions_df.count()} rows)")

print("\n--- Accounts (sample) ---")
display(accounts_df.limit(5))
print("\n--- Transactions (sample) ---")
display(transactions_df.limit(5))

In [0]:
CATALOG = dbutils.widgets.get('catalog')

def apply_comments(table_fqn, comments):
    for col, comment in comments.items():
        spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN `{col}` COMMENT '{comment}'")
    print(f"\u2714 {table_fqn} \u2014 {len(comments)} column comments applied")

apply_comments(f"{CATALOG}.finance.accounts", {
    "account_id":    "Unique identifier for the bank account",
    "customer_name": "Full name of the account holder (generated via Faker)",
    "account_type":  "Type of account: Checking, Savings, Credit Card, Investment, Loan, Money Market, or CD",
    "balance":       "Current account balance in the designated currency. Negative for Credit Card and Loan accounts",
    "currency":      "ISO currency code (USD, EUR, GBP, CAD, AUD, JPY, CHF)",
    "status":        "Account status: Active, Closed, Frozen, or Dormant",
    "opened_date":   "Date the account was opened (DateType)",
    "branch":        "Branch location where the account was opened (18 US cities)",
})

apply_comments(f"{CATALOG}.finance.transactions", {
    "transaction_id":   "Unique identifier for the transaction",
    "account_id":       "Foreign key referencing accounts.account_id",
    "transaction_date": "Timestamp of the transaction (TimestampType)",
    "amount":           "Transaction amount in the account currency. Always positive; direction indicated by transaction_type",
    "transaction_type": "Debit (money out) or Credit (money in)",
    "category":         "Transaction category: Payroll, Transfer, Purchase, Refund, Fee, Interest, Withdrawal, Deposit, Payment, or Dividend",
    "merchant":         "Merchant name for applicable transactions (40+ merchants); NULL for non-merchant txns",
    "channel":          "Channel used: Online, Branch, ATM, Mobile App, or Wire",
    "currency":         "ISO currency code inherited from the account",
    "description":      "Free-text description with reference number",
    "fraud_suspected":  "Pattern-based fraud flag. Higher probability for: large amounts >$2K, unusual channel-category combos, large withdrawals",
})

print(f"\n\u2705 All column comments applied for finance schema")

In [0]:
%sql
COMMENT ON SCHEMA IDENTIFIER(:catalog || '.finance') IS
'Finance sample dataset with realistic statistical distributions and Faker-generated PII. Entity table: `accounts` (~2K rows). Event table: `transactions` (100K-500K rows). Key features: 7 account types, 7 currencies, 40+ merchants, pattern-based fraud flags.';

In [0]:
CATALOG = dbutils.widgets.get('catalog')
remove_after_value = "2026-12-31"

spark.sql(f"ALTER SCHEMA `{CATALOG}`.`finance` SET TAGS ('RemoveAfter' = '{remove_after_value}')")
print(f"✔ RemoveAfter tag applied to finance schema ({remove_after_value})")